# Twitch recommendation data

Collects user context, followed channels and streams, recent VODs, and current live-stream candidates from Twitch Helix. It derives ranking-ready features without creating embeddings yet.

## Setup

Set `TWITCH_CLIENT_ID` and `TWITCH_CLIENT_SECRET` in `.env.local` or the notebook environment. Public streams and VODs use an app token. Set `TWITCH_USER_ACCESS_TOKEN` with the `user:read:follows` scope to collect followed channels and followed live streams; `TWITCH_USER_ID` is optional.

In [13]:
import math
import os
from pathlib import Path
from typing import Any

import pandas as pd
import requests

try:
    from dotenv import load_dotenv
    root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    load_dotenv(root / ".env.local")
    load_dotenv(root / ".env")
except ImportError:
    pass

CLIENT_ID = os.environ.get("TWITCH_CLIENT_ID")
CLIENT_SECRET = os.environ.get("TWITCH_CLIENT_SECRET")
USER_ACCESS_TOKEN = os.environ.get("TWITCH_USER_ACCESS_TOKEN")
USER_ID = os.environ.get("TWITCH_USER_ID")
API_BASE = "https://api.twitch.tv/helix"

if not CLIENT_ID or not CLIENT_SECRET:
    raise RuntimeError("Set TWITCH_CLIENT_ID and TWITCH_CLIENT_SECRET before running this notebook.")

In [14]:
def get_app_access_token() -> str:
    response = requests.post(
        "https://id.twitch.tv/oauth2/token",
        params={"client_id": CLIENT_ID, "client_secret": CLIENT_SECRET, "grant_type": "client_credentials"},
        timeout=30,
    )
    response.raise_for_status()
    return response.json()["access_token"]

APP_ACCESS_TOKEN = get_app_access_token()

def twitch_get(path: str, params: dict[str, Any] | list[tuple[str, str]] | None = None, user_token: bool = False) -> dict[str, Any]:
    token = USER_ACCESS_TOKEN if user_token else APP_ACCESS_TOKEN
    if not token:
        raise RuntimeError(f"{path} requires TWITCH_USER_ACCESS_TOKEN")
    response = requests.get(
        f"{API_BASE}{path}",
        headers={"Authorization": f"Bearer {token}", "Client-Id": CLIENT_ID},
        params=params or {},
        timeout=30,
    )
    if not response.ok:
        raise RuntimeError(f"Twitch API {response.status_code} for {path}: {response.text[:500]}")
    return response.json()

def fetch_pages(path: str, params: dict[str, Any], max_pages: int = 5, user_token: bool = False) -> list[dict[str, Any]]:
    params = dict(params)
    rows = []
    for _ in range(max_pages):
        payload = twitch_get(path, params, user_token)
        rows.extend(payload.get("data", []))
        cursor = payload.get("pagination", {}).get("cursor")
        if not cursor:
            break
        params["after"] = cursor
    return rows

## Connect a Twitch user account from the notebook

Clerk login does not create a Twitch API token. This cell uses Twitch's authorization-code flow directly: it opens Twitch in your browser, waits for the local callback, and keeps the returned user token in notebook memory. Add `http://localhost:8765/callback` as an OAuth redirect URL for the Twitch application, then run this cell and approve access. The requested scope is `user:read:follows`.

In [15]:
import secrets
import urllib.parse
import webbrowser
from http.server import BaseHTTPRequestHandler, HTTPServer

NOTEBOOK_REDIRECT_URI = os.environ.get("TWITCH_NOTEBOOK_REDIRECT_URI", "http://localhost:8765/callback")
redirect = urllib.parse.urlparse(NOTEBOOK_REDIRECT_URI)
if redirect.hostname not in {"localhost", "127.0.0.1"}:
    raise ValueError("TWITCH_NOTEBOOK_REDIRECT_URI must point to localhost for this notebook flow.")

oauth_state = secrets.token_urlsafe(32)
authorize_url = "https://id.twitch.tv/oauth2/authorize?" + urllib.parse.urlencode({
    "client_id": CLIENT_ID,
    "redirect_uri": NOTEBOOK_REDIRECT_URI,
    "response_type": "code",
    "scope": "user:read:follows",
    "state": oauth_state,
})

callback_result = {}

class CallbackHandler(BaseHTTPRequestHandler):
    def do_GET(self):
        query = urllib.parse.parse_qs(urllib.parse.urlparse(self.path).query)
        callback_result.update({key: values[0] for key, values in query.items() if values})
        self.send_response(200)
        self.send_header("Content-Type", "text/html")
        self.end_headers()
        self.wfile.write(b"<h2>Twitch authorization received.</h2><p>You can return to the notebook.</p>")

    def log_message(self, _format, *_args):
        del _format, _args

server = HTTPServer((redirect.hostname, redirect.port or 80), CallbackHandler)
print(f"Opening Twitch authorization in your browser: {authorize_url}")
webbrowser.open(authorize_url)
server.handle_request()
server.server_close()

if callback_result.get("state") != oauth_state:
    raise RuntimeError("Twitch OAuth state did not match; authorization was rejected.")
if "error" in callback_result:
    raise RuntimeError(f"Twitch OAuth failed: {callback_result.get('error_description', callback_result['error'])}")

token_response = requests.post(
    "https://id.twitch.tv/oauth2/token",
    params={
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
        "code": callback_result["code"],
        "grant_type": "authorization_code",
        "redirect_uri": NOTEBOOK_REDIRECT_URI,
    },
    timeout=30,
)
token_response.raise_for_status()
USER_ACCESS_TOKEN = token_response.json()["access_token"]
USER_ID = twitch_get("/users", user_token=True)["data"][0]["id"]
print(f"Connected Twitch user {USER_ID}. Re-run the data collection cell below to fetch follows and VODs.")

Opening Twitch authorization in your browser: https://id.twitch.tv/oauth2/authorize?client_id=fgb4hkuga276cklzx16gb5r6s90kym&redirect_uri=http%3A%2F%2Flocalhost%3A8765%2Fcallback&response_type=code&scope=user%3Aread%3Afollows&state=uM_dGIRuis0405lsNu9YqspbB-W31-SREh7foKpofpE
Connected Twitch user 1564891906. Re-run the data collection cell below to fetch follows and VODs.


In [16]:
user = None
if USER_ACCESS_TOKEN:
    user = twitch_get("/users", user_token=True).get("data", [None])[0]
elif USER_ID:
    user = twitch_get("/users", {"id": USER_ID}).get("data", [None])[0]
if user:
    USER_ID = user["id"]

users_df = pd.DataFrame([user] if user else [])
channels_df = pd.DataFrame(fetch_pages("/channels/followed", {"user_id": USER_ID, "first": 100}, user_token=True)) if USER_ACCESS_TOKEN and USER_ID else pd.DataFrame()
followed_streams_df = pd.DataFrame(fetch_pages("/streams/followed", {"user_id": USER_ID, "first": 100}, user_token=True)) if USER_ACCESS_TOKEN and USER_ID else pd.DataFrame()

vod_params = {"first": 100, "period": "month", "sort": "time", "type": "all"}
if USER_ID:
    vod_params["user_id"] = USER_ID
elif os.environ.get("TWITCH_VOD_GAME_ID"):
    vod_params["game_id"] = os.environ["TWITCH_VOD_GAME_ID"]

vods_df = pd.DataFrame(fetch_pages("/videos", vod_params)) if "user_id" in vod_params or "game_id" in vod_params else pd.DataFrame()
if vods_df.empty and "user_id" not in vod_params and "game_id" not in vod_params:
    print("Skipping VOD collection: set TWITCH_USER_ID, TWITCH_USER_ACCESS_TOKEN, or TWITCH_VOD_GAME_ID.")
print({"users": len(users_df), "followed_channels": len(channels_df), "followed_streams": len(followed_streams_df), "vods": len(vods_df)})

{'users': 1, 'followed_channels': 2, 'followed_streams': 1, 'vods': 0}


In [17]:
live_stream_page_limit = int(os.environ.get("TWITCH_LIVE_STREAM_PAGES", "50"))
live_streams_df = pd.DataFrame(fetch_pages("/streams", {"first": 100}, max_pages=live_stream_page_limit))
stream_user_ids = live_streams_df["user_id"].dropna().astype(str).unique().tolist() if "user_id" in live_streams_df else []
live_users = []
for start in range(0, len(stream_user_ids), 100):
    live_users.extend(twitch_get("/users", [("id", value) for value in stream_user_ids[start:start + 100]]).get("data", []))
live_users_df = pd.DataFrame(live_users)
print(f"Current live candidate streams: {len(live_streams_df)} from {live_stream_page_limit * 100} requested slots")

Current live candidate streams: 5000 from 5000 requested slots


## Derived features and baseline ranking

Category, channel, tags, and title fields remain available for future embeddings. The explicit weights below are a replaceable baseline, not a trained model.

In [18]:
streams = live_streams_df.copy()
if not streams.empty:
    now = pd.Timestamp.now(tz="UTC")
    streams["started_at_ts"] = pd.to_datetime(streams["started_at"], utc=True, errors="coerce")
    streams["stream_age_hours"] = (now - streams["started_at_ts"]).dt.total_seconds() / 3600
    streams["viewer_count"] = pd.to_numeric(streams["viewer_count"], errors="coerce").fillna(0)
    streams["title"] = streams["title"].fillna("")
    streams["game_name"] = streams["game_name"].fillna("")
    streams["tag_count"] = streams["tags"].apply(lambda value: len(value) if isinstance(value, list) else 0)
    streams["title_length"] = streams["title"].str.len()
    streams["viewer_log1p"] = streams["viewer_count"].map(math.log1p)
    followed = set(channels_df.get("broadcaster_id", pd.Series(dtype=str)).astype(str))
    recent_categories = set(vods_df.get("game_name", pd.Series(dtype=str)).dropna().astype(str))
    streams["is_followed_channel"] = streams["user_id"].astype(str).isin(followed)
    streams["matches_recent_vod_category"] = streams["game_name"].isin(recent_categories)
feature_columns = ["id", "user_id", "user_login", "user_name", "game_id", "game_name", "title", "language", "tags", "is_mature", "viewer_count", "started_at", "stream_age_hours", "viewer_log1p", "tag_count", "title_length", "is_followed_channel", "matches_recent_vod_category"]
stream_features_df = streams[[column for column in feature_columns if column in streams.columns]].copy()

base_weights = {"viewer_log1p": 0.45, "tag_count": 0.05, "title_length": 0.05}
additional_weights = {"is_followed_channel": 0.30, "matches_recent_vod_category": 0.15}
top_k = 25

def min_max(series: pd.Series) -> pd.Series:
    spread = series.max() - series.min()
    return (series - series.min()) / spread if spread else series * 0

ranked_streams_df = stream_features_df.copy()
if not ranked_streams_df.empty:
    ranked_streams_df["viewer_log1p_norm"] = min_max(ranked_streams_df["viewer_log1p"])
    ranked_streams_df["tag_count_norm"] = min_max(ranked_streams_df["tag_count"])
    ranked_streams_df["title_length_norm"] = min_max(ranked_streams_df["title_length"])
    ranked_streams_df["recommendation_score"] = (
        base_weights["viewer_log1p"] * ranked_streams_df["viewer_log1p_norm"]
        + base_weights["tag_count"] * ranked_streams_df["tag_count_norm"]
        + base_weights["title_length"] * ranked_streams_df["title_length_norm"]
        + additional_weights["is_followed_channel"] * ranked_streams_df["is_followed_channel"].astype(float)
        + additional_weights["matches_recent_vod_category"] * ranked_streams_df["matches_recent_vod_category"].astype(float)
    )
    ranked_streams_df = ranked_streams_df.sort_values("recommendation_score", ascending=False).head(top_k)
ranked_streams_df[["id", "user_name", "game_name", "title", "viewer_count", "recommendation_score"]]

,id,user_name,game_name,title,viewer_count,recommendation_score
80,317778536695,NiceWigg,Apex Legends,(DROPS) ALGS Split 2 Las Vegas | ELIMINATION D...,3595,0.608537
1,318190806372,Kamet0,League of Legends,KARMINE CORP 0 vs 0 MOVISTAR KOI | WORLDS 2026...,66380,0.516208
0,318188581220,Caedrel,League of Legends,🔴WORLDS 2026 KOI VS KC ELIMINATION🔴,96025,0.497500
2,318190614756,ibai,League of Legends,#WORLDS2026 | MKOI vs KC | EL PARTIDO DEL AÑO ...,46662,0.471288
3,319022894951,Trymacs,Minecraft,100.000 TNT Exploion Craft Attack Rekord aufst...,30193,0.431707
6,318194337507,otplol_,League of Legends,WORLDS - KC vs MKOI | LYON vs LOS | HLE vs BLG...,24021,0.401784
4,320982904412,Warframe,Warframe,"PMCF Quest Finale | Saturday, October 31 @ 12P...",24999,0.401344
11,317135893973,afoninje,Dota 2,World Squad [0:0] CIS Squad | Bo1 | PARI UNIVE...,17955,0.401298
5,317170747222,EDISON,Just Chatting,🎃КОНКУРС КОСТЮМОВ НА ХЕЛЛОУИН! [сейчас],24479,0.401143
10,318163325154,WarThunder_Esports,War Thunder,Invictus 1-2 VOYAGER | 4v4 Ground | WTCS Pro L...,19147,0.399834


In [19]:
# Optional reproducible snapshots for later model experiments.
output_dir = Path("data")
output_dir.mkdir(exist_ok=True)
frames = {
    "users": users_df,
    "channels": channels_df,
    "followed_streams": followed_streams_df,
    "vods": vods_df,
    "live_streams_raw": live_streams_df,
    "live_stream_features": stream_features_df,
    "ranked_live_streams": ranked_streams_df,
}
for name, frame in frames.items():
    frame.to_json(output_dir / f"{name}.jsonl", orient="records", lines=True, date_format="iso")
print(f"Wrote {len(frames)} JSONL snapshots to {output_dir.resolve()}")

Wrote 7 JSONL snapshots to /Users/edwynzhou/Documents/repos/TwitchEE/notebooks/data


## Export the collected data to Postgres

The export uses `DATABASE_URL` from `.env.local` and writes append-only snapshots to `twitch_recommendation_records`. Each row stores a dataset name, Twitch record key, collection timestamp, and the complete row as JSONB. This keeps raw API fields and derived features available for later embedding and model experiments without changing the app's existing user tables.

Install the Python Postgres driver in the notebook environment first if needed:

```python
%pip install "psycopg[binary]"
```

In [20]:
import importlib
import json
from datetime import datetime, timezone

psycopg = importlib.import_module("psycopg")
Jsonb = importlib.import_module("psycopg.types.json").Jsonb

DATABASE_URL = os.environ.get("DATABASE_URL")
if not DATABASE_URL:
    raise RuntimeError("Set DATABASE_URL in .env.local before exporting to Postgres.")

collection_time = datetime.now(timezone.utc)

# Dataset names map to the data contract used by the recommendation pipeline.
export_frames = {
    "users": (users_df, "id"),
    "channels": (channels_df, "broadcaster_id"),
    "followed_streams": (followed_streams_df, "id"),
    "vods": (vods_df, "id"),
    "live_streams_raw": (live_streams_df, "id"),
    "live_stream_features": (stream_features_df, "id"),
    "ranked_live_streams": (ranked_streams_df, "id"),
}

create_table_sql = """
CREATE TABLE IF NOT EXISTS twitch_recommendation_records (
    dataset TEXT NOT NULL,
    record_key TEXT NOT NULL,
    collected_at TIMESTAMPTZ NOT NULL,
    payload JSONB NOT NULL,
    PRIMARY KEY (dataset, record_key, collected_at)
)
"""
insert_sql = """
INSERT INTO twitch_recommendation_records (dataset, record_key, collected_at, payload)
VALUES (%s, %s, %s, %s)
ON CONFLICT (dataset, record_key, collected_at)
DO UPDATE SET payload = EXCLUDED.payload
"""

rows_to_insert = []
for dataset, (frame, key_column) in export_frames.items():
    if frame.empty:
        continue
    # JSON round-tripping converts pandas timestamps and missing values safely.
    records = json.loads(frame.to_json(orient="records", date_format="iso"))
    for index, record in enumerate(records):
        record_key = str(record.get(key_column) or f"row-{index}")
        rows_to_insert.append((dataset, record_key, collection_time, Jsonb(record)))

with psycopg.connect(DATABASE_URL) as connection:
    with connection.cursor() as cursor:
        cursor.execute(create_table_sql)
        cursor.executemany(insert_sql, rows_to_insert)

print(f"Exported {len(rows_to_insert)} records across {len(export_frames)} datasets at {collection_time.isoformat()}")

Exported 10029 records across 7 datasets at 2026-10-31T16:55:04.338747+00:00
